# 💼 Freelancer Income & Experience — Statistics Notebook
**Dataset:** 1,000 simulated freelancers across tech disciplines

**Variables:** Experience, Hourly Rate, Weekly Hours, Client Rating, Country, Skill, Remote Status, Monthly Income

This notebook applies the same statistical workflow as the Student Performance analysis:
EDA → Descriptive Stats → Normal Distribution → Z-Score Visualization → Correlation & Boxplots

In [ ]:
# Load necessary packages
import pandas as pd
import numpy as np

In [ ]:
# Generate synthetic freelancer dataset (1000 records)
np.random.seed(42)
n = 1000

countries = ['USA', 'UK', 'Germany', 'Canada', 'Australia', 'Netherlands', 'UAE', 'India']
skills = ['Motion Design', 'Web Development', 'Data Science', 'UX/UI Design', 'Copywriting', 'Video Editing', 'DevOps', 'AI/ML']

experience_years = np.random.randint(0, 15, n)
hourly_rate = np.round(20 + experience_years * 5 + np.random.normal(0, 12, n), 2).clip(15, 200)
weekly_hours = np.random.randint(10, 60, n)
client_rating = np.round(np.random.uniform(3.0, 5.0, n), 2)
monthly_income = np.round(hourly_rate * weekly_hours * 4.33, 2)
remote = np.random.choice(['Yes', 'No'], n, p=[0.78, 0.22])
country = np.random.choice(countries, n)
skill = np.random.choice(skills, n)

mydata = pd.DataFrame({
    'Freelancer_ID': range(1, n+1),
    'Skill': skill,
    'Country': country,
    'Experience_Years': experience_years,
    'Hourly_Rate_USD': hourly_rate,
    'Weekly_Hours': weekly_hours,
    'Client_Rating': client_rating,
    'Remote': remote,
    'Monthly_Income_USD': monthly_income
})

mydata.to_csv('freelancer_data.csv', index=False)
print('Dataset created:', mydata.shape)

In [ ]:
mydata.tail(5)

In [ ]:
mydata.head(10)

## 🔍 Basic Exploration
Shape, data types, and missing values — always check before doing any analysis.

In [ ]:
print('Shape:', mydata.shape)
print()
print('Data Types:')
print(mydata.dtypes)
print()
print('Missing Values:')
print(mydata.isnull().sum())

In [ ]:
print('=== Skill Distribution ===')
print(mydata['Skill'].value_counts())
print()
print('=== Country Distribution ===')
print(mydata['Country'].value_counts())
print()
print('=== Remote Work ===')
print(mydata['Remote'].value_counts())

## 📊 Descriptive Statistics
Central tendency, spread, and distribution shape for all numeric columns.

In [ ]:
mydata.describe().round(2)

In [ ]:
income = mydata['Monthly_Income_USD']

print('=== Monthly Income (USD) ===')
print(f'Mean:     ${income.mean():,.2f}')
print(f'Median:   ${income.median():,.2f}')
print(f'Std Dev:  ${income.std():,.2f}')
print(f'Min:      ${income.min():,.2f}')
print(f'Max:      ${income.max():,.2f}')
print(f'Range:    ${income.max() - income.min():,.2f}')
print(f'Skewness: {income.skew():.4f}')
print(f'Kurtosis: {income.kurtosis():.4f}')

## 🔔 Normal Distribution — Hourly Rate
Plotting the distribution of freelancer **Hourly Rates** using a normal distribution with KDE overlay.

The bell curve shape shows how rates cluster around the mean.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st

x = np.random.normal(loc=65, scale=18, size=10000)

sns.set_style('ticks')

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(x, kde=True, color='steelblue', ax=ax, stat='density')

ax.set_title('Normal Distribution — Freelancer Hourly Rate (USD)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hourly Rate (USD)')
ax.set_ylabel('Density')

print(f'Mean:  ${x.mean():.2f}')
print(f'Std:   ${x.std():.2f}')

plt.tight_layout()
plt.show()

## 📐 Z-Score Visualization
Where does a **$95/hr** freelancer fall relative to the market?

**Formula:** Z = (X - μ) / σ

- 🟠 Orange = Mean (μ)
- 🟢 Green dashed = ±1σ, ±2σ, ±3σ
- 🟣 Purple = Target freelancer rate ($95/hr)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st

x_i = 95    # target freelancer hourly rate
mu = 65     # population mean
sigma = 18  # population std dev

x = np.random.normal(mu, sigma, 10000)

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(x, color='lightsteelblue', stat='density', ax=ax)

ax.axvline(mu, color='orange', linewidth=2, label=f'Mean (μ = ${mu})')

for v in [-3, -2, -1, 1, 2, 3]:
    ax.axvline(mu + v * sigma, color='olivedrab', linewidth=1, linestyle='--')
    ax.text(mu + v * sigma, ax.get_ylim()[1] * 0.02, f'{v}σ',
            ha='center', fontsize=8, color='olivedrab')

ax.axvline(x_i, color='purple', linewidth=2.5, label=f'Target (${x_i}/hr)')

z = (x_i - mu) / sigma
p = st.norm.cdf(z)

ax.set_title(
    f'Z-Score Visualization — Hourly Rate\n'
    f'Z = ({x_i} - {mu}) / {sigma} = {z:.2f}  |  Percentile: {p*100:.1f}%',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Hourly Rate (USD)')
ax.set_ylabel('Density')
ax.legend()
ax.set_xlim(0, 140)

print(f'Z-Score for ${x_i}/hr:  {z:.4f}')
print(f'Percentile:            {p*100:.2f}%')
print(f'Interpretation:        This freelancer earns more than {p*100:.1f}% of the market.')

plt.tight_layout()
plt.show()

## 📈 Correlation — Experience vs Hourly Rate
Does more experience translate to higher rates? Pearson correlation test.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = mydata['Experience_Years'].corr(mydata['Hourly_Rate_USD'])
print(f'Pearson Correlation (Experience vs Hourly Rate): {corr:.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
sns.regplot(
    data=mydata, x='Experience_Years', y='Hourly_Rate_USD',
    scatter_kws={'alpha': 0.3, 'color': 'steelblue'},
    line_kws={'color': 'orangered', 'linewidth': 2},
    ax=ax
)
ax.set_title(f'Experience vs Hourly Rate  |  r = {corr:.2f}', fontsize=13, fontweight='bold')
ax.set_xlabel('Years of Experience')
ax.set_ylabel('Hourly Rate (USD)')

plt.tight_layout()
plt.show()

## 🌍 Hourly Rate by Skill — Boxplot
Which skill commands the highest market rate? Median-sorted boxplot.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = mydata.groupby('Skill')['Hourly_Rate_USD'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=mydata, x='Skill', y='Hourly_Rate_USD', order=order, palette='Blues_d', ax=ax)

ax.set_title('Hourly Rate Distribution by Skill', fontsize=13, fontweight='bold')
ax.set_xlabel('Skill')
ax.set_ylabel('Hourly Rate (USD)')
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

print('\nMedian Hourly Rate by Skill:')
print(mydata.groupby('Skill')['Hourly_Rate_USD'].median().sort_values(ascending=False).round(2))